# LeetCode 821 — Shortest Distance to a Character bằng Python

**Link đề:** https://leetcode.com/problems/shortest-distance-to-a-character/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, String

```python
class Solution:
    def shortestToChar(self, s: str, c: str) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu "khoảng cách tới ký tự `c` **gần nhất**" cho **mọi** vị trí trong chuỗi.
2. Đi từ brute force `O(n²)` → gom vị trí của `c` → tìm nhị phân `O(n log k)` → **hai lượt quét trái/phải** `O(n)`.
3. Học kỹ thuật **"nhớ vị trí gần nhất đã thấy"** khi quét một chiều, rồi ghép hai chiều bằng `min`.
4. Biết dùng các công cụ Python: `[0] * n`, `abs`, `enumerate`, `range` chạy ngược, `float('inf')`, `bisect`.
5. Trace, chứng minh bằng invariant, test biên và luyện các biến thể (bài 2) cùng họ "nearest to the left/right".

## 1. Đề bài và ví dụ

Cho chuỗi `s` và một ký tự `c` (chắc chắn xuất hiện trong `s`). Trả về mảng `answer` dài bằng `s`, trong đó `answer[i]` là **khoảng cách nhỏ nhất** từ vị trí `i` tới một vị trí `j` có `s[j] == c`. Khoảng cách giữa hai chỉ số là `abs(i - j)`.

**Ví dụ 1:** `s = "loveleetcode"`, `c = "e"` → `[3,2,1,0,1,0,0,1,2,2,1,0]`

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| s[i] | l | o | v | **e** | l | **e** | **e** | t | c | o | d | **e** |
| answer | 3 | 2 | 1 | 0 | 1 | 0 | 0 | 1 | 2 | 2 | 1 | 0 |

Chữ `e` ở các vị trí `3, 5, 6, 11`. Ví dụ `i = 8`: `e` gần nhất là ở `6` → khoảng cách `2`. `i = 4`: hòa giữa `3` và `5`, đều là `1`.

**Ví dụ 2:** `s = "aaab"`, `c = "b"` → `[3,2,1,0]`.

**Constraints:** `1 <= len(s) <= 10^4`; `s[i]` và `c` là chữ cái thường; `c` xuất hiện ít nhất một lần trong `s`.

> Ta cần tìm gì? Với **mỗi** `i`: `min(abs(i - j))` trên mọi `j` mà `s[j] == c`.

In [1]:
s = "loveleetcode"
c = "e"
for i, ch in enumerate(s):
    print(i, ch, "<- đây là c" if ch == c else "")

0 l 
1 o 
2 v 
3 e <- đây là c
4 l 
5 e <- đây là c
6 e <- đây là c
7 t 
8 c 
9 o 
10 d 
11 e <- đây là c


## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi, chỉ số và `enumerate`

Chuỗi trong Python giống một dãy ký tự: `s[0]` là ký tự đầu, `s[len(s) - 1]` (hay `s[-1]`) là ký tự cuối. `enumerate(s)` cho ra **cặp** `(chỉ số, ký tự)` để không phải tự đếm.

In [2]:
s = "aaab"
print(len(s), s[0], s[3], s[-1])
for i, ch in enumerate(s):
    print(f"i={i}, ch={ch!r}")

4 a b b
i=0, ch='a'
i=1, ch='a'
i=2, ch='a'
i=3, ch='b'


### 2.2. `abs()` — khoảng cách giữa hai chỉ số

`abs(x)` là giá trị tuyệt đối. `abs(i - j)` không quan tâm `j` ở trái hay phải `i`.

In [3]:
print(abs(0 - 3), abs(8 - 6), abs(4 - 5), abs(-7))

3 2 1 7


### 2.3. Tạo list độ dài `n`: `[0] * n` và list comprehension

- `[0] * n` tạo list gồm `n` số `0` — dùng làm "mảng kết quả" rồi điền dần.
- `[biểu_thức for x in dãy]` (list comprehension) tạo list mới bằng cách tính biểu thức cho từng phần tử.

In [4]:
n = 5
ans = [0] * n
print(ans)
ans[2] = 7
print(ans)
squares = [i * i for i in range(n)]
print(squares)
positions = [i for i, ch in enumerate("loveleetcode") if ch == "e"]
print("vị trí của 'e':", positions)

[0, 0, 0, 0, 0]
[0, 0, 7, 0, 0]
[0, 1, 4, 9, 16]
vị trí của 'e': [3, 5, 6, 11]


### 2.4. `range` chạy ngược

`range(start, stop, step)` sinh dãy từ `start` tới **trước** `stop`, bước `step`. Muốn đi từ `n-1` về `0`: `range(n - 1, -1, -1)` — `stop = -1` vì `stop` không được lấy.

In [5]:
n = 5
print(list(range(n)))
print(list(range(n - 1, -1, -1)))
print(list(reversed(range(n))))   # cách viết khác, cùng kết quả

[0, 1, 2, 3, 4]
[4, 3, 2, 1, 0]
[4, 3, 2, 1, 0]


### 2.5. "Vô cực" làm giá trị khởi tạo: `float('inf')`

Khi tìm **min**, ta cần một giá trị ban đầu lớn hơn mọi khoảng cách có thể. `float('inf')` lớn hơn mọi số. Một mẹo khác cho bài này: dùng `n` (độ dài chuỗi) vì khoảng cách thật luôn `< n`.

In [6]:
INF = float('inf')
print(INF > 10**18, min(INF, 3), min(5, 3))
print(INF - 2)   # vẫn là inf

True 3 3
inf


## 3. Cách 1 — Brute force: với mỗi `i`, quét toàn bộ chuỗi

Cách tự nhiên nhất: với mỗi `i`, duyệt mọi `j`; nếu `s[j] == c` thì cập nhật `best = min(best, abs(i - j))`.

- Có `n` vị trí `i`, mỗi `i` duyệt `n` vị trí `j` ⇒ `n²` phép so sánh.
- `n = 10^4` ⇒ `10^8` bước: với Python là **rất chậm** (có thể Time Limit Exceeded).

In [7]:
def shortest_brute(s: str, c: str) -> list[int]:
    n = len(s)
    ans = []
    for i in range(n):
        best = float('inf')
        for j in range(n):
            if s[j] == c:
                best = min(best, abs(i - j))
        ans.append(best)
    return ans

print(shortest_brute("loveleetcode", "e"))
print(shortest_brute("aaab", "b"))

[3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
[3, 2, 1, 0]


### Đếm số bước để thấy nút thắt

Đếm số lần chạy vòng trong cho `n` khác nhau — số bước tăng theo **bình phương**.

In [8]:
def count_brute_steps(n: int) -> int:
    steps = 0
    for i in range(n):
        for j in range(n):
            steps += 1
    return steps

for n in [10, 100, 1000]:
    print(f"n={n:>5} -> {count_brute_steps(n):>9} bước")
print("n=10^4 -> khoảng", 10**4 * 10**4, "bước (quá chậm)")

n=   10 ->       100 bước
n=  100 ->     10000 bước
n= 1000 ->   1000000 bước
n=10^4 -> khoảng 100000000 bước (quá chậm)


## 4. Cách 2 — Gom vị trí của `c` trước, rồi tìm nhị phân

Quan sát: chỉ những `j` có `s[j] == c` mới quan trọng. Gom chúng vào list `pos` (đã **tăng dần** vì ta duyệt trái → phải).

- Nếu mỗi `i` duyệt hết `pos` (dài `k`) ⇒ `O(n·k)`; tệ nhất `k ≈ n` vẫn `O(n²)`.
- Vì `pos` **đã sắp xếp**, ta dùng **tìm nhị phân** để tìm vị trí `c` đầu tiên `>= i` ⇒ chỉ cần so với nó và phần tử ngay trước nó.

### 4.1. Module `bisect`

`bisect.bisect_left(a, x)` trả về **chỗ chèn** `x` vào list đã sắp xếp `a` sao cho vẫn sắp xếp — tức là chỉ số của phần tử đầu tiên `>= x`. Chạy `O(log k)`.

In [9]:
import bisect
pos = [3, 5, 6, 11]
for x in [0, 4, 5, 8, 12]:
    k = bisect.bisect_left(pos, x)
    right = pos[k] if k < len(pos) else None
    left = pos[k - 1] if k > 0 else None
    print(f"x={x:>2}: chèn ở k={k}, c gần trái={left}, c gần phải(>=x)={right}")

x= 0: chèn ở k=0, c gần trái=None, c gần phải(>=x)=3
x= 4: chèn ở k=1, c gần trái=3, c gần phải(>=x)=5
x= 5: chèn ở k=1, c gần trái=3, c gần phải(>=x)=5
x= 8: chèn ở k=3, c gần trái=6, c gần phải(>=x)=11
x=12: chèn ở k=4, c gần trái=11, c gần phải(>=x)=None


In [10]:
import bisect

def shortest_bisect(s: str, c: str) -> list[int]:
    pos = [j for j, ch in enumerate(s) if ch == c]   # tăng dần
    ans = []
    for i in range(len(s)):
        k = bisect.bisect_left(pos, i)
        best = float('inf')
        if k < len(pos):          # có c ở vị trí >= i
            best = min(best, pos[k] - i)
        if k > 0:                 # có c ở vị trí < i
            best = min(best, i - pos[k - 1])
        ans.append(best)
    return ans

print(shortest_bisect("loveleetcode", "e"))
print(shortest_bisect("aaab", "b"))

[3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
[3, 2, 1, 0]


Cách 2 chạy `O(n + n log k)` — đã nhanh. Nhưng có một cách **gọn hơn và tuyến tính** không cần thư viện, chỉ dựa trên một quan sát rất đẹp.

## 5. Nút thắt và insight — "gần nhất bên trái" và "gần nhất bên phải"

Với mỗi `i`, ký tự `c` gần nhất chỉ có thể là:
- `c` **gần nhất bên trái** (vị trí `<= i`), gọi là `L(i)`, hoặc
- `c` **gần nhất bên phải** (vị trí `>= i`), gọi là `R(i)`.

Mọi `c` khác đều xa hơn một trong hai cái này. Vậy:

```text
answer[i] = min(i - L(i), R(i) - i)
```

**Điểm then chốt:** `L(i)` tính được bằng **một lượt quét trái → phải**, chỉ cần nhớ biến `prev` = vị trí `c` gần nhất vừa đi qua. Tương tự `R(i)` bằng một lượt quét phải → trái. Không cần lưu cả danh sách vị trí!

Ví dụ `s = "loveleetcode"`, `c = "e"`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| i − L(i) | ∞ | ∞ | ∞ | 0 | 1 | 0 | 0 | 1 | 2 | 3 | 4 | 0 |
| R(i) − i | 3 | 2 | 1 | 0 | 1 | 0 | 0 | 4 | 3 | 2 | 1 | 0 |
| min | 3 | 2 | 1 | 0 | 1 | 0 | 0 | 1 | 2 | 2 | 1 | 0 |

In [11]:
s, c = "loveleetcode", "e"
n = len(s)
prev = None
left_dist = []
for i in range(n):
    if s[i] == c:
        prev = i
    left_dist.append(i - prev if prev is not None else float('inf'))
print("i - L(i):", left_dist)

i - L(i): [inf, inf, inf, 0, 1, 0, 0, 1, 2, 3, 4, 0]


Ở trên `prev = None` nghĩa là "chưa thấy `c` nào bên trái". Để tránh phải kiểm tra `None`, mẹo hay dùng: khởi tạo `prev = -n` (hoặc `-10**9`) — một vị trí "ảo" rất xa bên trái, khiến `i - prev` luôn lớn hơn mọi khoảng cách thật, nên chắc chắn không được chọn bởi `min` (vì `c` tồn tại, lượt quét kia sẽ cho giá trị thật nhỏ hơn).

In [12]:
s, c = "loveleetcode", "e"
n = len(s)
prev = -n
print([ (i - (prev := (i if s[i] == c else prev))) for i in range(n) ])

[12, 13, 14, 0, 1, 0, 0, 1, 2, 3, 4, 0]


> Cell trên dùng toán tử `:=` (walrus — gán ngay trong biểu thức) chỉ để minh họa ngắn; trong lời giải chính ta viết vòng `for` thường cho dễ đọc. Giá trị `12` ở đầu là "khoảng cách ảo" `i - (-n)` = `0 + 12`, lớn hơn mọi khoảng cách thật.

## 6. Lời giải chính — hai lượt quét (từng đoạn rồi ghép)

### Đoạn 1: lượt quét trái → phải

`prev` = vị trí `c` gần nhất **bên trái hoặc tại** `i`. Ghi `ans[i] = i - prev`.

```python
prev = -n
for i in range(n):
    if s[i] == c:
        prev = i
    ans[i] = i - prev
```

### Đoạn 2: lượt quét phải → trái

`prev` bây giờ = vị trí `c` gần nhất **bên phải hoặc tại** `i`. Lấy `min` với giá trị đã có.

```python
prev = 2 * n
for i in range(n - 1, -1, -1):
    if s[i] == c:
        prev = i
    ans[i] = min(ans[i], prev - i)
```

`prev = 2 * n` là vị trí ảo rất xa bên phải (để `prev - i >= n + 1` > mọi khoảng cách thật).

**Thứ tự quan trọng:** cập nhật `prev` **trước** khi tính khoảng cách, để vị trí có `s[i] == c` nhận khoảng cách `0`.

In [13]:
def shortest_two_pass(s: str, c: str) -> list[int]:
    n = len(s)
    ans = [0] * n

    prev = -n                      # c "ảo" rất xa bên trái
    for i in range(n):             # trái -> phải
        if s[i] == c:
            prev = i
        ans[i] = i - prev

    prev = 2 * n                   # c "ảo" rất xa bên phải
    for i in range(n - 1, -1, -1): # phải -> trái
        if s[i] == c:
            prev = i
        ans[i] = min(ans[i], prev - i)

    return ans

print(shortest_two_pass("loveleetcode", "e"))
print(shortest_two_pass("aaab", "b"))

[3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
[3, 2, 1, 0]


## 7. TRACE từng bước

### 7.1. Bảng trace cho `s = "aaba"`, `c = "b"` (n = 4)

Lượt 1 (trái → phải), `prev` bắt đầu `-4`:

| i | s[i] | prev sau cập nhật | ans[i] = i − prev |
|---|---|---|---|
| 0 | a | -4 | 4 (ảo) |
| 1 | a | -4 | 5 (ảo) |
| 2 | **b** | 2 | 0 |
| 3 | a | 2 | 1 |

Lượt 2 (phải → trái), `prev` bắt đầu `8`:

| i | s[i] | prev sau cập nhật | prev − i | ans[i] = min(cũ, mới) |
|---|---|---|---|---|
| 3 | a | 8 | 5 | min(1, 5) = 1 |
| 2 | **b** | 2 | 0 | 0 |
| 1 | a | 2 | 1 | min(5, 1) = 1 |
| 0 | a | 2 | 2 | min(4, 2) = 2 |

Kết quả `[2, 1, 0, 1]`. Chú ý vị trí 0, 1: lượt 1 cho giá trị "ảo" lớn, lượt 2 sửa lại thành giá trị thật.

In [14]:
def trace_shortest(s: str, c: str) -> list[int]:
    n = len(s)
    ans = [0] * n
    print(f"s={s!r}, c={c!r}, n={n}")
    print("--- Lượt 1: trái -> phải ---")
    prev = -n
    for i in range(n):
        if s[i] == c:
            prev = i
        ans[i] = i - prev
        print(f"i={i} s[i]={s[i]} prev={prev:>3} ans[i]={ans[i]:>2}  ans={ans}")
    print("--- Lượt 2: phải -> trái ---")
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        old = ans[i]
        ans[i] = min(old, prev - i)
        print(f"i={i} s[i]={s[i]} prev={prev:>3} min({old}, {prev - i}) = {ans[i]}  ans={ans}")
    print("Kết quả:", ans)
    return ans

trace_shortest("aaba", "b")

s='aaba', c='b', n=4
--- Lượt 1: trái -> phải ---
i=0 s[i]=a prev= -4 ans[i]= 4  ans=[4, 0, 0, 0]
i=1 s[i]=a prev= -4 ans[i]= 5  ans=[4, 5, 0, 0]
i=2 s[i]=b prev=  2 ans[i]= 0  ans=[4, 5, 0, 0]
i=3 s[i]=a prev=  2 ans[i]= 1  ans=[4, 5, 0, 1]
--- Lượt 2: phải -> trái ---
i=3 s[i]=a prev=  8 min(1, 5) = 1  ans=[4, 5, 0, 1]
i=2 s[i]=b prev=  2 min(0, 0) = 0  ans=[4, 5, 0, 1]
i=1 s[i]=a prev=  2 min(5, 1) = 1  ans=[4, 1, 0, 1]
i=0 s[i]=a prev=  2 min(4, 2) = 2  ans=[2, 1, 0, 1]
Kết quả: [2, 1, 0, 1]


[2, 1, 0, 1]

### 7.2. Trace ví dụ đề (nhiều `c` liên tiếp và hòa khoảng cách)

In [15]:
trace_shortest("loveleetcode", "e");

s='loveleetcode', c='e', n=12
--- Lượt 1: trái -> phải ---
i=0 s[i]=l prev=-12 ans[i]=12  ans=[12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
i=1 s[i]=o prev=-12 ans[i]=13  ans=[12, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
i=2 s[i]=v prev=-12 ans[i]=14  ans=[12, 13, 14, 0, 0, 0, 0, 0, 0, 0, 0, 0]
i=3 s[i]=e prev=  3 ans[i]= 0  ans=[12, 13, 14, 0, 0, 0, 0, 0, 0, 0, 0, 0]
i=4 s[i]=l prev=  3 ans[i]= 1  ans=[12, 13, 14, 0, 1, 0, 0, 0, 0, 0, 0, 0]
i=5 s[i]=e prev=  5 ans[i]= 0  ans=[12, 13, 14, 0, 1, 0, 0, 0, 0, 0, 0, 0]
i=6 s[i]=e prev=  6 ans[i]= 0  ans=[12, 13, 14, 0, 1, 0, 0, 0, 0, 0, 0, 0]
i=7 s[i]=t prev=  6 ans[i]= 1  ans=[12, 13, 14, 0, 1, 0, 0, 1, 0, 0, 0, 0]
i=8 s[i]=c prev=  6 ans[i]= 2  ans=[12, 13, 14, 0, 1, 0, 0, 1, 2, 0, 0, 0]
i=9 s[i]=o prev=  6 ans[i]= 3  ans=[12, 13, 14, 0, 1, 0, 0, 1, 2, 3, 0, 0]
i=10 s[i]=d prev=  6 ans[i]= 4  ans=[12, 13, 14, 0, 1, 0, 0, 1, 2, 3, 4, 0]
i=11 s[i]=e prev= 11 ans[i]= 0  ans=[12, 13, 14, 0, 1, 0, 0, 1, 2, 3, 4, 0]
--- Lượt 2: phải -> trái ---
i=11 s[i]=e p

Quan sát: ở `i = 4` lượt 1 cho `1` (từ `e` ở 3), lượt 2 cho `1` (từ `e` ở 5) — hòa, `min` vẫn là `1`. Ở `i = 9` lượt 1 cho `3`, lượt 2 cho `2` → chọn `2`.

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def shortestToChar(self, s: str, c: str) -> list[int]:
        n = len(s)
        ans = [0] * n
        prev = -n
        for i in range(n):
            if s[i] == c:
                prev = i
            ans[i] = i - prev
        prev = 2 * n
        for i in range(n - 1, -1, -1):
            if s[i] == c:
                prev = i
            ans[i] = min(ans[i], prev - i)
        return ans

sol = Solution()
assert sol.shortestToChar("loveleetcode", "e") == [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
assert sol.shortestToChar("aaab", "b") == [3, 2, 1, 0]
print("Solution PASS 2 ví dụ đề")

Solution PASS 2 ví dụ đề


## 9. Vì sao thuật toán đúng?

**Invariant lượt 1:** sau khi xử lý vị trí `i`, `prev` là vị trí lớn nhất `j <= i` có `s[j] == c` (hoặc `-n` nếu chưa có). Do đó `ans[i] = i - prev` = khoảng cách tới `c` gần nhất **bên trái** (hoặc một số `> n - 1` nếu không có).

**Invariant lượt 2:** sau khi xử lý `i` (đi từ phải), `prev` là vị trí nhỏ nhất `j >= i` có `s[j] == c` (hoặc `2n`). `prev - i` = khoảng cách tới `c` gần nhất **bên phải**.

**Ghép:** với `c` bất kỳ ở vị trí `j`:
- nếu `j <= i` thì `i - j >= i - L(i)` (vì `L(i)` là `c` bên trái lớn nhất);
- nếu `j >= i` thì `j - i >= R(i) - i`.

Vậy `min` của hai giá trị là khoảng cách nhỏ nhất thật sự. Giá trị ảo không bao giờ được chọn: `c` có trong `s` nên ít nhất một phía cho khoảng cách thật `<= n - 1`, còn giá trị ảo luôn `>= n`.

In [17]:
n = 4
print("giá trị ảo nhỏ nhất lượt 1:", 0 - (-n), "| lượt 2:", 2 * n - (n - 1), "| khoảng cách thật lớn nhất:", n - 1)

giá trị ảo nhỏ nhất lượt 1: 4 | lượt 2: 5 | khoảng cách thật lớn nhất: 3


### 9.1. Độ phức tạp

Gọi `n = len(s)`, `k` = số lần `c` xuất hiện.

| Cách | Time | Space phụ (ngoài output) |
|---|---|---|
| Brute force | `O(n²)` | `O(1)` |
| Gom vị trí + `bisect` | `O(n log k)` | `O(k)` |
| Hai lượt quét | `O(n)` — đúng `2n` bước | `O(1)` |

Mảng `ans` dài `n` là output bắt buộc nên thường không tính vào bộ nhớ phụ.

## 10. Cách 4 — Một lượt, hai con trỏ trên danh sách vị trí

Một ý tưởng "two pointers" khác: gom `pos` (vị trí của `c`), rồi duyệt `i` từ trái sang phải cùng con trỏ `k` chỉ vào `c` **bên phải gần nhất**. Khi `i` vượt qua `pos[k]`, tăng `k`. Vì `i` chỉ tăng nên `k` cũng chỉ tăng — tổng cộng `O(n + k)`.

In [18]:
def shortest_pointer(s: str, c: str) -> list[int]:
    pos = [j for j, ch in enumerate(s) if ch == c]
    ans = []
    k = 0                                  # pos[k] = c đầu tiên >= i
    for i in range(len(s)):
        while k < len(pos) and pos[k] < i:
            k += 1
        best = float('inf')
        if k < len(pos):
            best = pos[k] - i
        if k > 0:
            best = min(best, i - pos[k - 1])
        ans.append(best)
    return ans

print(shortest_pointer("loveleetcode", "e"))
print(shortest_pointer("aaab", "b"))

[3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
[3, 2, 1, 0]


### Code nộp LeetCode — phiên bản con trỏ

In [19]:
class SolutionPointer:
    def shortestToChar(self, s: str, c: str) -> list[int]:
        pos = [j for j, ch in enumerate(s) if ch == c]
        ans = []
        k = 0
        for i in range(len(s)):
            while k < len(pos) and pos[k] < i:
                k += 1
            best = pos[k] - i if k < len(pos) else float('inf')
            if k > 0:
                best = min(best, i - pos[k - 1])
            ans.append(best)
        return ans

print(SolutionPointer().shortestToChar("loveleetcode", "e"))

[3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]


### 10.1. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi cặp `(i, j)` | `O(n²)` | `O(1)` | Hiểu đề, làm oracle để test |
| 2. `bisect` | tìm nhị phân trong `pos` | `O(n log k)` | `O(k)` | Khi có nhiều truy vấn rời rạc |
| 3. Hai lượt quét | `min(trái, phải)` | `O(n)` | `O(1)` | **Lời giải chuẩn**, ngắn gọn |
| 4. Con trỏ trên `pos` | `k` chỉ tăng | `O(n)` | `O(k)` | Luyện two pointers |

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ đề.
2. Chuỗi 1 ký tự (`"a"`, `c="a"`).
3. `c` chỉ ở đầu / chỉ ở cuối (chỉ một phía có giá trị thật).
4. Toàn bộ là `c`.
5. `c` cách đều, hòa khoảng cách.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.shortestToChar("loveleetcode", "e"), [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0])
check("đề 2", sol.shortestToChar("aaab", "b"), [3, 2, 1, 0])
check("1 ký tự", sol.shortestToChar("a", "a"), [0])
check("c ở đầu", sol.shortestToChar("baaa", "b"), [0, 1, 2, 3])
check("toàn c", sol.shortestToChar("eeee", "e"), [0, 0, 0, 0])
check("hòa", sol.shortestToChar("abaab", "b"), [1, 0, 1, 1, 0])
check("c giữa", sol.shortestToChar("xxcxx", "c"), [2, 1, 0, 1, 2])

PASS đề 1: [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
PASS đề 2: [3, 2, 1, 0]
PASS 1 ký tự: [0]
PASS c ở đầu: [0, 1, 2, 3]
PASS toàn c: [0, 0, 0, 0]
PASS hòa: [1, 0, 1, 1, 0]
PASS c giữa: [2, 1, 0, 1, 2]


### 11.1. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên (có seed)

Brute force chậm nhưng **chắc chắn đúng** → dùng làm "oracle" để kiểm tra các cách nhanh.

In [21]:
import random
random.seed(821)
fns = [shortest_brute, shortest_bisect, shortest_two_pass, shortest_pointer,
       Solution().shortestToChar, SolutionPointer().shortestToChar]
for _ in range(400):
    n = random.randint(1, 15)
    s = "".join(random.choice("abc") for _ in range(n))
    c = random.choice(s)                     # bảo đảm c có trong s
    expected = shortest_brute(s, c)
    for f in fns:
        assert f(s, c) == expected, (s, c, f)
print("400 bộ ngẫu nhiên: 6 hàm cho cùng kết quả")

400 bộ ngẫu nhiên: 6 hàm cho cùng kết quả


### 11.2. Thử hiệu năng với `n = 10^4`

In [22]:
import time
s = "".join(random.choice("abcdefghij") for _ in range(10**4)) + "z"
t0 = time.perf_counter(); r1 = shortest_two_pass(s, "z"); t1 = time.perf_counter()
r2 = shortest_bisect(s, "z"); t2 = time.perf_counter()
assert r1 == r2
print(f"hai lượt: {t1 - t0:.4f}s | bisect: {t2 - t1:.4f}s | ans[0] = {r1[0]}")

hai lượt: 0.0015s | bisect: 0.0023s | ans[0] = 10000


## 12. Lỗi người mới hay gặp

### Lỗi 1: chỉ quét một chiều

```python
prev = -n
for i in range(n):
    if s[i] == c: prev = i
    ans[i] = i - prev      # thiếu lượt phải -> trái
```

Vì sao sai: các vị trí **trước** `c` đầu tiên nhận giá trị ảo, và vị trí gần `c` bên phải hơn bị tính sai.

In [23]:
def wrong_one_pass(s, c):
    n = len(s); ans = [0] * n; prev = -n
    for i in range(n):
        if s[i] == c:
            prev = i
        ans[i] = i - prev
    return ans

print("sai :", wrong_one_pass("aaab", "b"))
print("đúng:", shortest_two_pass("aaab", "b"))

sai : [4, 5, 6, 0]
đúng: [3, 2, 1, 0]


### Lỗi 2: tính khoảng cách **trước** khi cập nhật `prev`

```python
ans[i] = i - prev
if s[i] == c: prev = i     # đảo thứ tự
```

Vì sao sai: tại vị trí có `c`, khoảng cách phải là `0`, nhưng ta lại dùng `prev` cũ.

In [24]:
def wrong_order(s, c):
    n = len(s); ans = [0] * n
    prev = -n
    for i in range(n):
        ans[i] = i - prev
        if s[i] == c:
            prev = i
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        ans[i] = min(ans[i], prev - i)
        if s[i] == c:
            prev = i
    return ans

print("sai :", wrong_order("loveleetcode", "e"))
print("đúng:", shortest_two_pass("loveleetcode", "e"))

sai : [3, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 5]
đúng: [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]


### Lỗi 3: khởi tạo `prev = 0` thay vì vị trí ảo

`prev = 0` nghĩa là "giả vờ có `c` ở vị trí 0" → các vị trí trước `c` đầu tiên nhận khoảng cách nhỏ **sai**, và lượt 2 không sửa được vì `min` giữ giá trị nhỏ sai đó.

In [25]:
def wrong_init(s, c):
    n = len(s); ans = [0] * n
    prev = 0                                  # SAI
    for i in range(n):
        if s[i] == c:
            prev = i
        ans[i] = i - prev
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans

print("sai :", wrong_init("aaab", "b"))
print("đúng:", shortest_two_pass("aaab", "b"))

sai : [0, 1, 1, 0]
đúng: [3, 2, 1, 0]


### Lỗi 4: `range(n - 1, 0, -1)` bỏ sót vị trí 0

`stop` không được lấy, nên muốn tới `0` phải viết `stop = -1`.

In [26]:
print(list(range(3, 0, -1)), "<- thiếu 0")
print(list(range(3, -1, -1)), "<- đúng")

[3, 2, 1] <- thiếu 0
[3, 2, 1, 0] <- đúng


## 13. Mẹo nhận dạng pattern

- Đề hỏi "với **mỗi** vị trí, tìm phần tử thỏa điều kiện **gần nhất**" → nghĩ tới **quét hai chiều**: một lượt mang thông tin từ trái, một lượt từ phải, rồi ghép.
- Khoảng cách tới "gần nhất" = `min(gần nhất bên trái, gần nhất bên phải)`.
- Họ bài tương tự: *Product of Array Except Self* (tích trái × tích phải), *Trapping Rain Water* (max trái, max phải), *Candy* (ràng buộc trái/phải), *Distance to nearest 0* trong ma trận (BFS — phiên bản 2 chiều).
- Khi có list vị trí **đã sắp xếp** và cần "phần tử gần nhất với x" → `bisect`.

## 14. Biến thể

### 14.1. Khoảng cách tới **bất kỳ** ký tự trong một tập

Thay `s[i] == c` bằng `s[i] in targets`. Toàn bộ lập luận vẫn giữ nguyên.

In [27]:
def shortest_to_any(s: str, targets: set) -> list[int]:
    n = len(s); ans = [0] * n
    prev = -n
    for i in range(n):
        if s[i] in targets:
            prev = i
        ans[i] = i - prev
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if s[i] in targets:
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans

print(shortest_to_any("leetcode", set("aeiou")))

[1, 0, 0, 1, 1, 0, 1, 0]


### 14.2. Trả về **chỉ số** của `c` gần nhất (hòa thì chọn bên trái)

Thay vì lưu khoảng cách, ta lưu vị trí bên trái và bên phải rồi so sánh.

In [28]:
def nearest_index(s: str, c: str) -> list[int]:
    n = len(s)
    left = [None] * n
    prev = None
    for i in range(n):
        if s[i] == c:
            prev = i
        left[i] = prev
    res = [0] * n
    prev = None
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        L, R = left[i], prev
        if R is None or (L is not None and i - L <= R - i):
            res[i] = L
        else:
            res[i] = R
    return res

print(nearest_index("loveleetcode", "e"))

[3, 3, 3, 3, 3, 5, 6, 6, 6, 11, 11, 11]


### 14.3. Phiên bản số: khoảng cách tới số 0 gần nhất trong mảng

Đây chính là bài 821 với `s` là list số và `c = 0` — cùng một code, chỉ đổi điều kiện.

In [29]:
def dist_to_zero(nums: list[int]) -> list[int]:
    n = len(nums); ans = [0] * n
    prev = -n
    for i in range(n):
        if nums[i] == 0:
            prev = i
        ans[i] = i - prev
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if nums[i] == 0:
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans

print(dist_to_zero([5, 0, 3, 4, 0, 7, 8]))

[1, 0, 1, 1, 0, 1, 2]


## 15. Bài tập (tự làm, KHÔNG xem đáp án trước)

### Bài 1 — Tính tay
Với `s = "abcba"`, `c = "b"`: ghi giá trị `ans` sau lượt 1, sau lượt 2.

### Bài 2 — Dự đoán output
Không chạy code, đoán `list(range(4, -1, -2))` và `[0] * 3 + [1]`.

### Bài 3 — Điền TODO cho lời giải hai lượt quét

In [30]:
def ex3_shortest(s: str, c: str) -> list[int]:
    n = len(s)
    ans = [0] * n
    prev = -n
    for i in range(n):
        # TODO: cập nhật prev nếu s[i] == c, rồi ghi ans[i]
        pass
    prev = 2 * n
    # TODO: lượt phải -> trái, lấy min
    return ans

# Tự làm xong rồi mới mở test:
# assert ex3_shortest("loveleetcode", "e") == [3,2,1,0,1,0,0,1,2,2,1,0]

### Bài 4 — Chỉ số `c` gần nhất bên phải
Viết hàm trả về list `nxt` với `nxt[i]` = chỉ số `j >= i` nhỏ nhất có `s[j] == c`, hoặc `-1` nếu không có.

In [31]:
def ex4_next_index(s: str, c: str) -> list[int]:
    # TODO: một lượt quét phải -> trái
    pass

# assert ex4_next_index("abcab", "a") == [0, 3, 3, 3, -1]

### Bài 5 — Biến thể: khoảng cách tới nguyên âm gần nhất
Cho `s` (chữ thường, có ít nhất một nguyên âm `aeiou`). Trả về khoảng cách từ mỗi vị trí tới nguyên âm gần nhất.

In [32]:
def ex5_vowel_distance(s: str) -> list[int]:
    # TODO
    pass

# assert ex5_vowel_distance("strength") == [3, 2, 1, 0, 1, 2, 3, 4]

### Bài 6 — Mini challenge: chỗ ngồi xa người nhất
Cho list `seats` gồm `0` (trống) và `1` (có người), có ít nhất một `0` và một `1`. Tìm khoảng cách **lớn nhất** tới người gần nhất mà một người mới có thể đạt được khi ngồi vào một ghế trống (LeetCode 849 — Maximize Distance to Closest Person).

In [33]:
def ex6_max_dist_to_closest(seats: list[int]) -> int:
    # TODO: dùng lại ý tưởng hai lượt quét
    pass

# assert ex6_max_dist_to_closest([1, 0, 0, 0, 1, 0, 1]) == 2
# assert ex6_max_dist_to_closest([1, 0, 0, 0]) == 3
# assert ex6_max_dist_to_closest([0, 1]) == 1

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: làm theo đúng bảng ở mục 7.1.
- Bài 4: đi từ phải sang trái, nhớ "vị trí `c` gần nhất đã thấy".
- Bài 6: mỗi ghế trống có "khoảng cách tới người gần nhất" — đúng bài 821 với `c = 1`; đáp án là **max** của các giá trị đó tại ghế trống.

**Mức 2 — cấu trúc/cú pháp**
- Bài 3: `if s[i] == c: prev = i` rồi `ans[i] = i - prev`; lượt 2 dùng `range(n - 1, -1, -1)` và `min`.
- Bài 4: `prev = -1` ban đầu; `nxt = [0] * n`.
- Bài 5: thay điều kiện bằng `s[i] in "aeiou"`.

**Mức 3 — gần code**
- Bài 6: `d = dist_to(seats, 1)`; `return max(d[i] for i in range(n) if seats[i] == 0)`.

## 17. Đáp án tham khảo

**Bài 1:** sau lượt 1 (prev = -5 ban đầu): `[5, 0, 1, 0, 1]`; sau lượt 2: `[1, 0, 1, 0, 1]`.

**Bài 2:** `[4, 2, 0]` và `[0, 0, 0, 1]`.

In [34]:
print(list(range(4, -1, -2)), [0] * 3 + [1])
trace_shortest("abcba", "b");

[4, 2, 0] [0, 0, 0, 1]
s='abcba', c='b', n=5
--- Lượt 1: trái -> phải ---
i=0 s[i]=a prev= -5 ans[i]= 5  ans=[5, 0, 0, 0, 0]
i=1 s[i]=b prev=  1 ans[i]= 0  ans=[5, 0, 0, 0, 0]
i=2 s[i]=c prev=  1 ans[i]= 1  ans=[5, 0, 1, 0, 0]
i=3 s[i]=b prev=  3 ans[i]= 0  ans=[5, 0, 1, 0, 0]
i=4 s[i]=a prev=  3 ans[i]= 1  ans=[5, 0, 1, 0, 1]
--- Lượt 2: phải -> trái ---
i=4 s[i]=a prev= 10 min(1, 6) = 1  ans=[5, 0, 1, 0, 1]
i=3 s[i]=b prev=  3 min(0, 0) = 0  ans=[5, 0, 1, 0, 1]
i=2 s[i]=c prev=  3 min(1, 1) = 1  ans=[5, 0, 1, 0, 1]
i=1 s[i]=b prev=  1 min(0, 0) = 0  ans=[5, 0, 1, 0, 1]
i=0 s[i]=a prev=  1 min(5, 1) = 1  ans=[1, 0, 1, 0, 1]
Kết quả: [1, 0, 1, 0, 1]


In [35]:
def ans3_shortest(s: str, c: str) -> list[int]:
    n = len(s)
    ans = [0] * n
    prev = -n
    for i in range(n):
        if s[i] == c:
            prev = i
        ans[i] = i - prev
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans


def ans4_next_index(s: str, c: str) -> list[int]:
    n = len(s)
    nxt = [0] * n
    prev = -1
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        nxt[i] = prev
    return nxt


def ans5_vowel_distance(s: str) -> list[int]:
    n = len(s)
    ans = [0] * n
    prev = -n
    for i in range(n):
        if s[i] in "aeiou":
            prev = i
        ans[i] = i - prev
    prev = 2 * n
    for i in range(n - 1, -1, -1):
        if s[i] in "aeiou":
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans


def ans6_max_dist_to_closest(seats: list[int]) -> int:
    d = dist_to_zero([1 - x for x in seats])   # người (1) -> 0 để dùng lại hàm
    return max(d[i] for i in range(len(seats)) if seats[i] == 0)


assert ans3_shortest("loveleetcode", "e") == [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
assert ans3_shortest("aaab", "b") == [3, 2, 1, 0]
assert ans4_next_index("abcab", "a") == [0, 3, 3, 3, -1]
assert ans5_vowel_distance("strength") == [3, 2, 1, 0, 1, 2, 3, 4]
assert ans6_max_dist_to_closest([1, 0, 0, 0, 1, 0, 1]) == 2
assert ans6_max_dist_to_closest([1, 0, 0, 0]) == 3
assert ans6_max_dist_to_closest([0, 1]) == 1
print("Đáp án bài 3-6 PASS")

Đáp án bài 3-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

**Template "gần nhất trái/phải":**

```python
n = len(s); ans = [0] * n
prev = -n                         # ảo bên trái
for i in range(n):
    if OK(i): prev = i            # cập nhật TRƯỚC
    ans[i] = i - prev
prev = 2 * n                      # ảo bên phải
for i in range(n - 1, -1, -1):
    if OK(i): prev = i
    ans[i] = min(ans[i], prev - i)
```

**Checklist trước khi submit:**
- [ ] Có đủ **hai** lượt quét (trái → phải và phải → trái)?
- [ ] Cập nhật `prev` **trước** khi tính khoảng cách?
- [ ] Giá trị ảo đủ xa (`-n` và `2n`), không phải `0`?
- [ ] Vòng ngược dùng `range(n - 1, -1, -1)` (có vị trí 0)?
- [ ] Trả về list độ dài đúng `len(s)`, tên method `shortestToChar`?
- [ ] Đã test: `c` ở đầu, ở cuối, toàn `c`, chuỗi 1 ký tự?